In [ ]:
import torch
print("GPU доступен:", torch.cuda.is_available())
print("Устройство:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

In [ ]:
!pip install ultralytics roboflow -q

In [ ]:
from roboflow import Roboflow
from google.colab import userdata

api_key = userdata.get("ROBOFLOW_API_KEY")
rf = Roboflow(api_key=api_key)
project = rf.workspace("testcarplate").project("russian-license-plates-classification-by-this-type")
version = project.version(1)
dataset = version.download("yolov8")

print("Датасет скачан в:", dataset.location)

In [ ]:
import os

dataset_path = dataset.location
print("Содержимое папки датасета:")
for item in os.listdir(dataset_path):
    print(" ", item)

# Смотрим data.yaml — там описаны классы и пути к данным
yaml_path = os.path.join(dataset_path, "data.yaml")
with open(yaml_path, "r") as f:
    print("\nСодержимое data.yaml:")
    print(f.read())

In [ ]:
train_path = os.path.join(dataset_path, "train", "images")
val_path   = os.path.join(dataset_path, "valid", "images")
test_path  = os.path.join(dataset_path, "test",  "images")

def count_images(path):
    if os.path.exists(path):
        return len(os.listdir(path))
    return 0

print(f"Train: {count_images(train_path)} изображений")
print(f"Val:   {count_images(val_path)} изображений")
print(f"Test:  {count_images(test_path)} изображений")

## Параметры:
model   — yolov8n.pt  
data    — наш датасет  
epochs  — 30 эпох достаточно для дообучения  
imgsz   — размер изображения 640x640  
batch   — размер батча (16 комфортно для T4)  
patience— остановка если нет улучшений 10 эпох подряд  
project — папка для сохранения результатов  
name    — имя эксперимента  

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")  # скачается автоматически ~6MB

results = model.train(
    data=yaml_path,
    epochs=30,
    imgsz=640,
    batch=16,
    patience=10,
    project="runs/train",
    name="russian_plates",
    device=0,        # 0 = первый GPU
    verbose=True,
)

print("\nОбучение завершено!")

In [ ]:
from IPython.display import Image as IPImage, display
import glob

# Путь к результатам
results_dir = "/content/runs/detect/runs/train/russian_plates"

# Показываем график обучения
results_png = os.path.join(results_dir, "results.png")
if os.path.exists(results_png):
    display(IPImage(results_png, width=900))

# Показываем примеры предсказаний на валидации
val_pred = os.path.join(results_dir, "val_batch0_pred.jpg")
if os.path.exists(val_pred):
    display(IPImage(val_pred, width=900))

In [ ]:
best_weights = os.path.join(results_dir, "weights", "best.pt")
trained_model = YOLO(best_weights)

metrics = trained_model.val(data=yaml_path, split="test")

print(f"\nmAP50:    {metrics.box.map50:.3f}")
print(f"mAP50-95: {metrics.box.map:.3f}")
print(f"Precision:{metrics.box.mp:.3f}")
print(f"Recall:   {metrics.box.mr:.3f}")

In [ ]:
from google.colab import files

print(f"Скачиваем веса: {best_weights}")
files.download(best_weights)